In [9]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_isx"


In [12]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values(["variable", "firm_id"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
EIMS.IC,XSGA_DA,Income Statement,Depreciation and amortization,False,"Selected XSGA bucket from the prior step is 'Operatingexpenses' plus 'Salaries and related expenses'. The separate line 'Depreciation and amortization' appears at the same operating-expense level and before the broader total 'Operating Expense', so it appears outside the selected XSGA rows rather than embedded within them. Summary row chosen over component rows to avoid double counting."
HAPD.IC,XSGA_DA,Income Statement,Depreciation/Amortization in SGA,False,"Selected XSGA bucket consists of 'Payroll Expenses' and 'Selling and Marketing Expense', which do not include the separate SG&A D&A line. Use only the summary SG&A D&A row, not its components."
ICESEA.IC,XSGA_DA,Income Statement,Depreciation and amortisation,False,"The selected XSGA reference bucket is 'Operating expenses'. 'Depreciation and amortisation' is shown as a separate operating line rather than as an apparent sub-component under 'Operating expenses', so it should be added separately. Use only the summary row, not its components."
OCS.IC,XSGA_DA,Income Statement,Depreciation in Research & Development Expenses,True,"SG&A depreciation rows appear embedded in the selected parent row 'General and administrative expenses', so they are excluded. The selected XSGA R&D bucket consists of granular component rows rather than the total 'Research and development expenses' row, so the separate R&D depreciation lines appear to sit outside that selected bucket and should be added. Manual review is warranted because the earlier XSGA/XRD mapping notes reference the total R&D row, while the listed XSGA final_choice shows only R&D components."
OCS.IC,XSGA_DA,Income Statement,Depreciation of Financial Lease Right-of-Use Assets in Research & Development Expenses,True,"SG&A depreciation rows appear embedded in the selected parent row 'General and administrative expenses', so they are excluded. The selected XSGA R&D bucket consists of granular component rows rather than the total 'Research and development expenses' row, so the separate R&D depreciation lines appear to sit outside that selected bucket and should be added. Manual review is warranted because the earlier XSGA/XRD mapping notes reference the total R&D row, while the listed XSGA final_choice shows only R&D components."
SVN.IC,XSGA_DA,Income Statement,Depreciation,False,"Selected only the summary depreciation row to avoid double counting with the separate right-of-use asset depreciation component. The selected XSGA bucket consists only of labor and other operating expenses, so this separate depreciation line should be added separately."
